# 02 · Raw-material price forecasting benchmark (h = 3 / 6 / 9 months)

> **Public copy (NDA).** Outputs are cleared and private values are removed from the text. Run it locally with the NDA data in `data/`; never push it with outputs. Figures are not saved from this copy (`save` is a no-op): the public figures come from `public/reports/make_public_figures.py`.


**The question:** which approach is worth pursuing for raw-material price prediction?
**The method:** a *ladder* of models, each scored under the same leakage-safe rolling-origin backtest against the naive forecast.

| Tier | Model | Code |
|---|---|---|
| 0 | naive (last value), drift | `src/models.py::univariate_forecasts` |
| 1 | ETS (damped), ARIMA(1,1,0) + drift, Theta (all on log prices) | same |
| 2 | Dynamic regression: ridge with no intercept, own lags + known lagged drivers (`features.DRIVERS`) | `dynreg_predict` |
| 3 | Global LightGBM pooled over all 12 series (+ P10/P50/P90 quantile models) | `lgbm_predict` |
| 5 | Equal-weight ensemble of tiers 1–3, with online split-conformal 80% intervals | `backtest.add_ensemble`, `add_conformal` |

Design choices:
- **Target:** for each horizon, the cumulative log-change `log(y[t+h] / y[t])`, with one model per horizon.
- **Backtest origins:** monthly from Jan-2020 on an expanding window.
- **Publication lags:** every driver is shifted by `max(0, lag_driver − lag_target)` (see `features.PUB_LAGS`).
- **Main metric:** relative MAE versus naive. **A value below 1 beats the random walk.**

In [ ]:
import sys, warnings, time
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT)); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl
import src.features as fe, src.backtest as bt, src.models as mo

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
mpl.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.6,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.prop_cycle": mpl.cycler(color=PAL), "lines.linewidth": 1.6, "legend.frameon": False,
    "legend.fontsize": 8, "font.family": "sans-serif", "figure.dpi": 110,
})
DIV = mpl.colors.LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])  # blue = beats naive
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
save = lambda fig, name: None  # public copy: never write figures
CACHE = ROOT / "data" / "processed" / "backtest.parquet"
RERUN = True   # set False to reuse the cached backtest

T, X = fe.build_panel()
print("targets:", T.shape, "| drivers:", X.shape)
T.apply(lambda s: pd.Series({"first": s.first_valid_index().date(), "last": s.last_valid_index().date(),
                             "n": s.notna().sum(), "family": fe.FAMILY[s.name], "pub_lag_m": fe.PUB_LAGS.get(s.name)})).T

## 1 · Guard-rails: leakage test & reproduction of the notebook-01 baseline

In [ ]:
for h in (3, 6, 9):
    for origin in ("2020-06-01", "2022-03-01", "2025-01-01"):
        fe.check_no_leakage(T, X, h, origin)
print("✓ features at any origin are unchanged when all future data is scrambled")

## 2 · Run the backtest

In [ ]:
if RERUN or not CACHE.exists():
    t0 = time.time()
    res = bt.rolling_origin(T, X, horizons=(3, 6, 9), start="2020-01-01")
    res = bt.add_ensemble(res, ("ets", "arima", "theta", "dynreg", "lgbm"), "ensemble")
    res = bt.add_conformal(res, "ensemble", alpha=0.2)
    CACHE.parent.mkdir(exist_ok=True); res.to_parquet(CACHE)
    print(f"backtest: {time.time() - t0:.0f}s")
res = pd.read_parquet(CACHE)
print(res.shape, "| origins", res.origin.min().date(), "→", res.origin.max().date())

# sanity: naive MAPE must equal notebook 01 (§10) exactly
nb01 = {("S09", 3): 4.7, ("S10", 3): 4.5, ("S11", 3): 4.3, ("D01", 3): 7.1, ("D02", 9): 15.3, ("CRU_CR_DE", 6): 16.4}  # nda-ok: naive MAPE (%), not a price
naive_mape = bt.score(res[res.model == "naive"], by=("series", "h")).MAPE.round(1)
assert all(naive_mape[k] == v for k, v in nb01.items()), naive_mape
print("✓ naive MAPE reproduces notebook 01")

## 3 · Leaderboard

In [ ]:
sc = bt.score(res, by=("model", "h"))
board = sc[["MAPE", "relMAE_vs_naive", "dir_acc"]].unstack("h")
board.sort_values(("relMAE_vs_naive", 3))

In [ ]:
order_m = ["naive", "drift", "ets", "arima", "theta", "dynreg", "lgbm", "ensemble"]
order_s = ["S09", "S10", "S11", "CRU_SS304_DE", "SS304_ALLOY_DE", "CRU_CR_DE", "CRU_CR_FR", "CRU_HDG_DE", "D01", "D02", "S03", "S04"]
rel = bt.score(res, by=("series", "h", "model")).relMAE_vs_naive.unstack("model")[order_m]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), sharey=True)
for ax, h in zip(axes, (3, 6, 9)):
    m = rel.xs(h, level="h").loc[order_s]
    im = ax.imshow(m.values, cmap=DIV, vmin=0.6, vmax=1.4, aspect="auto")
    ax.set_xticks(range(len(order_m))); ax.set_xticklabels(order_m, rotation=45, ha="right")
    ax.set_yticks(range(len(order_s))); ax.set_yticklabels(order_s); ax.grid(False)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(j, i, f"{m.iat[i, j]:.2f}", ha="center", va="center", fontsize=6.5, color=INK)
    ax.set_title(f"h = {h} months")
fig.colorbar(im, ax=axes, shrink=0.8, label="MAE relative to naive (blue < 1 = better)")
fig.suptitle("Relative MAE vs random walk, by series × model", x=0.01, ha="left", fontweight="bold")
save(fig, "12_leaderboard_heatmap"); plt.show()

### 3.1 Are the wins statistically real?
The Diebold–Mariano test (Harvey-corrected, HAC variance with lag h−1) compares squared log errors against naive.
A negative statistic means the model is better; the test is two-sided.

In [ ]:
dm = pd.concat({m: bt.dm_test(res, m) for m in ["dynreg", "ensemble", "arima", "lgbm"]}, axis=1)
sig = dm.xs("p_value", axis=1, level=1).lt(0.05) & dm.xs("DM", axis=1, level=1).lt(0)
print("Series × h where the model is significantly better than naive (p < 0.05):")
print(sig.sum().to_string())
dm.xs("DM", axis=1, level=1).round(2).join(dm.xs("p_value", axis=1, level=1).round(3), lsuffix="_DM", rsuffix="_p").loc[["S09", "S10", "S11", "CRU_SS304_DE", "D01", "CRU_CR_DE"]]

### 3.2 Shock (2021-22) vs calm periods

In [ ]:
r2 = res.assign(window=bt.window_label(res))
win = bt.score(r2[r2.model.isin(["naive", "arima", "dynreg", "lgbm", "ensemble"])], by=("window", "model", "h"))
win[["MAPE", "relMAE_vs_naive"]].unstack("h").round(2)

### 3.3 Selection without hindsight
Choosing the "best model per series" on the full backtest is optimistic.
Instead, select on origins before 2023 and score on origins from 2023 onward.

In [ ]:
cut = pd.Timestamp("2023-01-01")
cand = ["naive", "arima", "ets", "theta", "dynreg", "lgbm", "ensemble"]
pre = bt.score(res[(res.origin < cut) & res.model.isin(cand)], by=("series", "h", "model")).relMAE_vs_naive.unstack("model")
pick = pre.idxmin(axis=1).rename("picked")
post_all = bt.score(res[(res.origin >= cut) & res.model.isin(cand)], by=("series", "h", "model")).relMAE_vs_naive.unstack("model")
honest = pd.DataFrame({"picked": pick, "relMAE_2023+": [post_all.loc[i, m] for i, m in pick.items()],
                       "ensemble_2023+": post_all["ensemble"]})
print("Mean relMAE of the pre-selected model on 2023+ data, by h:")
print(honest.groupby(level="h")[["relMAE_2023+", "ensemble_2023+"]].mean().round(3))
honest.unstack("h").round(2)

### 3.4 The market's own forecast: LME 3-month futures
For copper and aluminium, the 3-month LME price is a free, forward-looking forecast for h = 3.
The futures-implied log-change is `log(3M / cash)`, using monthly averages at the origin.

In [ ]:
import src.data_loading as dl
lme = dl.load_lme()
fut = {"D01": np.log(lme["Cu 3m USD/t"] / lme["Cu cash USD/t"]), "D02": np.log(lme["Al 3m USD/t"] / lme["Al cash USD/t"])}
rows = []
for s, f in fut.items():
    g = res[(res.series == s) & (res.h == 3) & (res.model.isin(["naive", "arima", "ensemble"]))]
    base = g[g.model == "naive"].copy(); base["model"] = "LME 3M futures"; base["pred"] = f.reindex(base.origin).values
    gg = pd.concat([g, base.dropna(subset=["pred"])])
    sc_f = bt.score(gg, by=("model",))
    rows.append(sc_f[["n", "MAPE", "relMAE_vs_naive", "dir_acc"]].assign(series=s))
pd.concat(rows).reset_index().set_index(["series", "model"]).round(3)

## 4 · Uncertainty: are the 80% intervals calibrated?

In [ ]:
cov = bt.score(res[res.model.isin(["lgbm", "ensemble"])], by=("model", "h"))[["cover_80", "pinball"]].unstack("h")
cov.index = ["ensemble + conformal", "LightGBM quantile (P10–P90)"]
cov

In [ ]:
c = bt.score(res[res.model == "ensemble"], by=("series", "h")).cover_80.unstack("h")
fig, ax = plt.subplots(figsize=(11, 3.4))
x = np.arange(len(c)); w = 0.27
for i, h in enumerate((3, 6, 9)):
    ax.bar(x + (i - 1) * w, c[h], width=w - 0.03, color=PAL[i], label=f"h = {h}")
ax.axhline(0.8, color=INK2, lw=1, ls="--"); ax.text(len(c) - 0.5, 0.81, "target 80%", fontsize=7.5, color=INK2, ha="right")
ax.set_xticks(x); ax.set_xticklabels(c.index, fontsize=7.5); ax.set_ylim(0, 1)
ax.set_title("Empirical coverage of the ensemble's conformal 80% interval"); ax.legend(ncol=3, loc="lower left")
save(fig, "13_interval_coverage"); plt.show()

## 5 · Why does dynamic regression win on the Eurostat PPIs?
Standardised ridge coefficients for S09 (the EU steel PPI) at the latest origin, h = 3.
The market price (CRU CR coil) and the market–PPI gap carry the signal. This is the 1-month lead found in the EDA.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
F3 = fe.make_features(T, X, 3).xs("S09", level="series")
cols = ["own_dlog_l0", "own_mom3", "own_dev_ma12"] + [c for c in F3 if any(c.startswith(f"x_{d}_") or c == f"x_{d}" for d in fe.DRIVERS["S09"])]
tr = F3.dropna(subset=cols + ["y"])
sc_ = StandardScaler().fit(tr[cols]); rd = Ridge(alpha=50, fit_intercept=False).fit(sc_.transform(tr[cols]), tr.y)
coef = pd.Series(rd.coef_, index=cols).sort_values()
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(coef.index, coef.values, color=[PAL[0] if v > 0 else PAL[7] for v in coef.values], height=0.6)
ax.axvline(0, color=AXIS, lw=0.8); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_title("S09 dynamic regression: standardised coefficients (h = 3)")
save(fig, "14_dynreg_s09_coefs"); plt.show()

## 6 · Production forecast from the latest data
The point forecast is the ensemble. The interval is P10–P90 from that series' realised backtest residuals at the same horizon (split-conformal).

In [ ]:
fc = bt.forecast_now(T, X, res)
view = fc[["series", "origin", "h", "y_origin", "p10", "p50", "p90", "chg_%"]].copy()
view["origin"] = view.origin.dt.strftime("%Y-%m")
view.set_index(["series", "h"]).round(1)

In [ ]:
# Saved for the NDA-safe public figure (public/reports/make_public_figures.py): quantiles + last observed value.
(fc.assign(y0=fc.series.map(lambda s: T[s].dropna().iloc[-1]), origin=fc.series.map(lambda s: T[s].dropna().index[-1]))
   .to_parquet(ROOT / "data" / "processed" / "forecast_fans.parquet"))
show = [("D01", "Copper (EUR/t)"), ("CRU_CR_DE", "CR coil DE (EUR/t)"), ("CRU_SS304_DE", "Stainless 304 DE (EUR/t)"), ("S09", "EU steel PPI")]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, (s, lab) in zip(axes, show):
    hist = T[s].dropna().loc["2023-01":]
    f = fc[fc.series == s].sort_values("h")
    xs = [hist.index[-1]] + list(f.target_month); y0 = hist.iloc[-1]
    ax.fill_between(xs, [y0] + list(f.p10), [y0] + list(f.p90), color=PAL[0], alpha=0.18, lw=0, label="P10–P90")
    ax.plot(hist.index, hist, color=INK2, lw=1.3, label="history")
    ax.plot(xs, [y0] + list(f.p50), color=PAL[0], lw=1.8, marker="o", ms=4, label="ensemble P50")
    ax.set_title(lab, fontsize=9.5); ax.tick_params(axis="x", rotation=30)
axes[0].legend(loc="upper left")
fig.suptitle("Forecast fans from the latest observation (h = 3, 6, 9 months)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); save(fig, "15_forecast_fans"); plt.show()

## 7 · From prices to product cost: a BOM cost forecast with joint uncertainty
- **Masses:** `features.bom_masses()`, the `data.zip` BOM. The RM6 carbon-steel mass is assumed (see DATA_OVERVIEW).
- **Plastics:** €2,500/t in 2021, moved with S11.
- **Uncertainty:** each simulation draws **one backtest origin** and applies the realised ensemble residuals of *all* materials at that origin together. This keeps the cross-material correlation; the copper and steel shocks of 2021-22 arrive together.

In [ ]:
mass = fe.bom_masses()
plast_eur_t = 2500 / T.S11.loc["2021"].mean()               # EUR/t per index point
to_eur_t = {s: (plast_eur_t if s == "S11" else 1.0) for s in mass.index}
rng = np.random.default_rng(42)
ens = res[res.model == "ensemble"].assign(r=lambda d: d.actual - d.pred)
rows = []
for h in (3, 6, 9):
    R = ens[ens.h == h].pivot_table(index="origin", columns="series", values="r")[mass.index].dropna()
    draws = R.iloc[rng.integers(0, len(R), 5000)].values                    # joint residual vectors
    f = fc[fc.h == h].set_index("series").loc[mass.index]
    p50 = f.y_origin * np.exp(f.ensemble) * pd.Series(to_eur_t)
    sims = (f.y_origin * pd.Series(to_eur_t)).values * np.exp(f.ensemble.values + draws)  # EUR/t per material
    now = (f.y_origin * pd.Series(to_eur_t))
    for prod in mass.columns:
        m = mass[prod].fillna(0) / 1000
        cost_sims = sims @ m.values
        rows.append(dict(product=prod, h=h, cost_now=float(now @ m), p10=np.percentile(cost_sims, 10),
                         p50=float(p50 @ m), p90=np.percentile(cost_sims, 90)))
bomf = pd.DataFrame(rows).set_index(["product", "h"])
bomf["p50_chg_%"] = 100 * (bomf.p50 / bomf.cost_now - 1)
bomf["p90_chg_%"] = 100 * (bomf.p90 / bomf.cost_now - 1)
print("Priced material cost per unit (EUR); 'now' = latest observation of each material")
bomf.round(1)

## 8 · Does forecasting create money? A forward-buy rule backtest
At each origin, compare buying 3 months of need *now*, with a carry cost of 0.4% per month, against buying at spot in 3 months.
The rule forward-buys only when the model's P50 at h = 3 is above +2%.
The benchmarks are **always spot**, **always forward**, and **perfect foresight**.
Results are in EUR per tonne bought, averaged over the backtest.

In [ ]:
def hedge(series, model, thr=0.02, carry=0.004, h=3):
    g = res[(res.series == series) & (res.model == model) & (res.h == h)].sort_values("origin")
    fwd = g.y_origin * (1 + carry) ** h
    spot = g.y_true
    rule = np.where(g.pred > thr, fwd, spot)
    oracle = np.minimum(fwd, spot)
    return pd.Series({"n": len(g), "always_spot": spot.mean(), "always_forward": fwd.mean(), "rule": rule.mean(),
                      "oracle": oracle.mean(), "rule_saving_vs_spot_%": 100 * (1 - rule.mean() / spot.mean()),
                      "oracle_saving_%": 100 * (1 - oracle.mean() / spot.mean()), "triggers": int((g.pred > thr).sum())})

hedge_tbl = pd.DataFrame({(s, m): hedge(s, m) for s in ["D01", "CRU_CR_DE", "CRU_SS304_DE", "S09", "S10"]
                          for m in ["ensemble", "dynreg", "arima"]}).T
hedge_tbl

## 9 · Conclusions
The key results are written up in [`DATA_OVERVIEW.md` §6](../DATA_OVERVIEW.md):

- naive is hard to beat on traded prices;
- the Eurostat PPIs are forecastable from the market prices that lead them;
- the conformal intervals are close to calibrated;
- forecast value comes from **risk sizing** more than from point accuracy.